In [2]:
import json
import pandas as pd



In [3]:

with open("../raw-data/userdata_mini.json", "r", encoding="utf-8") as f:
    raw_user_data = json.load(f)

with open("../raw-data/posts_mini.json", "r", encoding="utf-8") as f:
    posts = json.load(f)


In [ ]:

# def build_post_text_dict(posts, raw_user_data, csv_path):


#     pairs_df = pd.read_csv(csv_path)

#     post_text_dict = {}


#     target_post_ids = set(pairs_df["P_id"].dropna().unique())

#     for uri in target_post_ids:
#         post = posts.get(uri)
#         if post:
#             text = post.get("text")
#             if text:
#                 post_text_dict[uri] = text


#     user_ids = set(pairs_df["A_id"].dropna().unique()) | \
#                set(pairs_df["S_id"].dropna().unique())

#     for user_id in user_ids:
#         user_data = raw_user_data.get(user_id)
#         if not user_data:
#             continue

#         for activity in user_data.get("history", []):
#             uri = activity.get("post_uri")
#             text = activity.get("text")

#             if uri and text:
#                 post_text_dict[uri] = text

#     return post_text_dict


In [4]:
def build_post_text_dict(posts, raw_user_data):

    post_text_dict = {}

    # 1️⃣ Add all user history
    for user_data in raw_user_data.values():
        for activity in user_data.get("history", []):
            uri = activity.get("post_uri")
            text = activity.get("text")

            if uri and text:
                post_text_dict[uri] = text

    # 2️⃣ Add all original posts
    for uri, post in posts.items():
        text = post.get("text")
        if uri and text:
            post_text_dict[uri] = text

    return post_text_dict

In [5]:
all_text_dict = build_post_text_dict(posts,raw_user_data)

In [6]:
print(len(all_text_dict))

1682770


In [8]:
with open("../raw-data/posts_text.jsonl", "w") as f:
    for uri, text in all_text_dict.items():
        f.write(json.dumps({
            "post_uri": uri,
            "text": text
        }) + "\n")

print("JSONL created.")

JSONL created.


In [ ]:
 
import pandas as pd
from tqdm import tqdm
import gc
import torch
from pathlib import Path

import glob
import json

def save_text_feature_json(
):
    """
    Convert parquet feature chunks into a single JSON
    mapping: {post_uri: feature_dict}

    Memory-safe: streams chunk by chunk.
    """

    DIR = Path("../../data/processed/text_features")
    files = sorted(glob.glob(str(DIR / "part_*.parquet")))

    DIR.parent.mkdir(parents=True, exist_ok=True)

    with DIR.open("w", encoding="utf-8") as f:
        f.write("{\n")

        first = True

        for file in files:
            df_chunk = pd.read_parquet(file)

            for _, row in df_chunk.iterrows():
                post_uri = row["post_uri"]

                # remove raw text column if it exists
                row_dict = row.drop(labels=["text"], errors="ignore").to_dict()

                if not first:
                    f.write(",\n")
                first = False

                json.dump(post_uri, f)
                f.write(": ")
                json.dump(row_dict, f)

            del df_chunk

        f.write("\n}")

    print(f"Text feature dict written to {DIR}")

save_text_feature_json()

Text feature dict written to data/processed/text_features
